In [ ]:
!pip uninstall -y transformers
!pip install --no-cache-dir transformers

Found existing installation: transformers 5.0.0
Uninstalling transformers-5.0.0:
  Successfully uninstalled transformers-5.0.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 291.8 MB/s eta 0:00:00


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [ ]:
model_name = "Qwen/Qwen1.5-0.5B"

tokenizer = AutoTokenizer.from_pretrained(model_name)

actor = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=torch.float32
).to(device)

actor.train()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/1.24G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 1024)
    (layers): ModuleList(
      (0-23): 24 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=1024, out_features=1024, bias=True)
          (k_proj): Linear(in_features=1024, out_features=1024, bias=True)
          (v_proj): Linear(in_features=1024, out_features=1024, bias=True)
          (o_proj): Linear(in_features=1024, out_features=1024, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=1024, out_features=2816, bias=False)
          (up_proj): Linear(in_features=1024, out_features=2816, bias=False)
          (down_proj): Linear(in_features=2816, out_features=1024, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((1024,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((1024,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((1024,), eps=1e-06)
    (rot

In [ ]:
for param in actor.parameters():
    param.requires_grad = False

for param in actor.lm_head.parameters():
    param.requires_grad = True

In [ ]:
class ValueHead(nn.Module):
    def __init__(self, hidden_size):
        super().__init__()
        self.value = nn.Linear(hidden_size, 1)

    def forward(self, hidden_states):
        return self.value(hidden_states).mean()

critic = ValueHead(actor.config.hidden_size).to(device)

In [ ]:
optimizer = torch.optim.Adam(
    list(actor.lm_head.parameters()) + list(critic.parameters()),
    lr=1e-5
)

In [ ]:
clip_epsilon = 0.2
value_coef = 0.5
entropy_coef = 0.01

In [ ]:
def reward_function(text):
    reward = 0
    if "learning" in text.lower():
        reward += 1
    if len(text.split()) > 30:
        reward += 1
    return reward - 1

In [ ]:
def collect_trajectory(prompt):

    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    generated = actor.generate(**inputs, max_new_tokens=40)

    outputs = actor(generated, output_hidden_states=True)

    logits = outputs.logits
    hidden_states = outputs.hidden_states[-1]

    log_probs = F.log_softmax(logits, dim=-1)
    selected = log_probs.gather(2, generated.unsqueeze(-1)).squeeze(-1)
    log_prob = selected.mean()

    value = critic(hidden_states)

    text = tokenizer.decode(generated[0], skip_special_tokens=True)
    reward = torch.tensor(reward_function(text), dtype=torch.float32).to(device)

    return log_prob, value, reward, text

In [ ]:
def ppo_update(old_log_prob, old_value, reward, prompt):

    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    generated = actor.generate(**inputs, max_new_tokens=40)

    outputs = actor(generated, output_hidden_states=True)

    logits = outputs.logits
    hidden_states = outputs.hidden_states[-1]

    log_probs = F.log_softmax(logits, dim=-1)
    selected = log_probs.gather(2, generated.unsqueeze(-1)).squeeze(-1)
    new_log_prob = selected.mean()

    new_value = critic(hidden_states)

    advantage = reward - old_value.detach()

    ratio = torch.exp(new_log_prob - old_log_prob.detach())

    unclipped = ratio * advantage
    clipped = torch.clamp(ratio, 1 - clip_epsilon, 1 + clip_epsilon) * advantage

    policy_loss = -torch.min(unclipped, clipped)
    value_loss = value_coef * (reward - new_value).pow(2)

    probs = F.softmax(logits, dim=-1)
    entropy = -(probs * log_probs).sum(dim=-1).mean()
    entropy_loss = -entropy_coef * entropy

    loss = policy_loss + value_loss + entropy_loss

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    return loss.item()

In [ ]:
prompts = [
    "Explain reinforcement learning.",
    "What is GPU optimization?",
    "Describe comparative learning."
]

for epoch in range(2):
    print("\nEpoch:", epoch)

    for prompt in prompts:
        old_log_prob, old_value, reward, text = collect_trajectory(prompt)

        loss = ppo_update(old_log_prob, old_value, reward, prompt)

        print("Generated:", text[:80])
        print("Reward:", reward.item())
        print("Loss:", loss)
        print("------")

Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.



Epoch: 0


Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


Generated: Explain reinforcement learning. Reinforcement learning is a type of machine lear
Reward: 1.0
Loss: -0.4305686354637146
------


Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


...

Generated: Explain reinforcement learning. Reinforcement learning is a type of machine lear
Reward: 1.0
Loss: -0.4517762064933777
------


Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


Generated: What is GPU optimization? GPU optimization is a technique used to improve the pe
Reward: 0.0
Loss: -0.045363664627075195
------


Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


Generated: Describe comparative learning. The following is a list of comparative learning a
Reward: 1.0
Loss: -0.4518898129463196
------


In [ ]:
def generate(prompt):
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    output = actor.generate(**inputs, max_new_tokens=60)
    return tokenizer.decode(output[0], skip_special_tokens=True)

print(generate("Explain reinforcement learning."))

Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


Explain reinforcement learning. Reinforcement learning is a type of machine learning that involves training an agent to make decisions based on feedback from its environment. The agent learns to interact with the environment and make decisions that maximize its reward. The agent receives feedback in the form of rewards or penalties for its actions, and uses this feedback to
